In [ ]:
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' | tee /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install google-chrome-stable

OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [2,548 B]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:9 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,409 B]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [72.4 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [2,990 kB]
Get:13 http://security.ubuntu.com/ubunt

In [ ]:
!pip install selenium
!pip install webdriver-manager

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [ ]:
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time


def extract_stock_data(html):
    soup = BeautifulSoup(html, 'html.parser')

    date_element = soup.select_one("div[class^='ChartHeader_date']")
    date_text = date_element.get_text(strip=True)

    year, month, day = date_text.split('/')

    date = f'{int(year):04d}-{int(month):02d}-{int(day):02d}'

    items = soup.select("div[class^='ChartHeader_item']")

    values = {}

    for item in items:
        label = item.select_one("span[class^='ChartHeader_label']")
        value = item.select_one("span[class^='ChartHeader_value']")

        if label and value:
            values[label.get_text(strip=True)] = value.get_text(strip=True)

    stock_data = [
        date,
        values['始値'],
        values['高値'],
        values['安値'],
        values['終値']
    ]

    return stock_data


def get_stock_values(driver, url):

    driver.get(url)

    wait = WebDriverWait(driver, 30)

    wait.until(
        EC.frame_to_be_available_and_switch_to_it(
            (By.CSS_SELECTOR, "div[class^='Chart_container'] iframe")
        )
    )

    wait.until(
        EC.visibility_of_element_located(
            (By.CSS_SELECTOR, "div[class^='ChartHeader_row']")
        )
    )

    chart = wait.until(
        EC.visibility_of_element_located(
            (By.TAG_NAME, "svg")
        )
    )

    chart_width = chart.size['width']

    ActionChains(driver).move_to_element(chart).perform()


    ActionChains(driver).move_by_offset(
        chart_width // 2 - 1,
        0
    ).perform()

    stock_values = []

    for i in range(chart_width):

        html = driver.page_source

        try:
            stock_data = extract_stock_data(html)

            if stock_data not in stock_values:
                stock_values.append(stock_data)

        except (AttributeError, KeyError, ValueError):
            pass

        ActionChains(driver).move_by_offset(-1, 0).perform()

    return stock_values


chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')


service = Service(ChromeDriverManager().install())

chrome_driver = webdriver.Chrome(
    service=service,
    options=chrome_options
)


url = 'https://www.nikkei.com/markets/worldidx/chart/nk225/?type=6month'


start_time = time.time()

try:
    stock_values = get_stock_values(
        chrome_driver,
        url
    )

finally:
    chrome_driver.quit()

end_time = time.time()


for stock_data in stock_values:
    print(
        stock_data[0],
        stock_data[1],
        stock_data[2],
        stock_data[3],
        stock_data[4]
    )

print(
    'スクレイピング開始時間：',
    time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time))
)

print(
    'スクレイピング終了時間：',
    time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time))
)

print(
    f'スクレイピングにかかった時間：{end_time - start_time:.2f}秒'
)


2026-09-11 64276.82 64312.02 63208.63 64011.34
2026-09-10 64768.53 65270.95 64186.68 65270.95
2026-09-09 65087.22 65794.03 65017.68 65142.78
2026-09-08 65843.69 66791.84 65269.33 65269.33
2026-09-07 65600.42 66668.71 65600.42 66399.84
2026-09-04 64498.94 65182.45 64228.47 65020.94
2026-09-03 64724.81 64724.81 63772.8 64214.48
2026-09-02 65195.43 65195.43 64215.47 64325.64
2026-09-01 65885.47 66525.7 65576.61 66215.34
2026-08-31 65668.51 66311.93 64832.1 66311.93
2026-08-28 66104.49 66842.82 66012.2 66405.56
2026-08-27 66775.78 66954.69 65782.59 66131.98
2026-08-26 65604.49 66442.34 65390.55 66262.16
2026-08-25 65195.26 65904.24 64609.47 65856.43
2026-08-24 65978.95 66257.73 65470.95 65528.09
2026-08-21 65427.69 66125.88 65260.22 66016.36
2026-08-20 65732.79 66325.32 65652.25 66216.79
2026-08-19 66812.27 66833.51 65133.98 65326.42
2026-08-18 68847.35 69093.2 67368.96 67460.73
2026-08-17 68882.06 69225.57 68492.04 69220.25
2026-08-14 68811.17 69608.24 68471.15 68713.8
2026-08-13 68033.17